# 平衡网络去嵌入

演示*平衡*网络，即 2N 端口网络的去嵌入。

## 设置

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.media import CPW
from skrf.network import concat_ports, connect

rf.stylely()

In [ ]:
# base parameters
freq = rf.Frequency(1e-3,10,1001,'ghz')
cpw  = CPW(freq, w=0.6e-3, s=0.25e-3, ep_r=10.6, z0_port=50)

## 构建夹具网络* 一段短的阻抗不匹配的传输线，其输入端具有类似连接器的并联电容。* 添加一些串扰，并进行微调。

In [ ]:
"""
             l1
    0----+-=======-2
         |
         = c1
         |
        GND

             l1
    1----+-=======-3
         |
         = c1
         |
        GND
"""


l1 = cpw.line(20, unit='mm')
c1 = cpw.shunt_capacitor(C=0.15e-12)
l1 = connect(c1, 1, l1, 0)
li = concat_ports([l1, l1], port_order='second')
Fix = li
Fix.name = 'Fix'
Fix.nudge(1e-4)
Left = Fix
# flip fixture for right side
Right = Fix.flipped()

## 构建待测器件 (DUT) 网络* 一段长度不匹配的传输线* 通过调整参数添加一些串扰

In [ ]:
"""
        l2
    0-=======-2
        l2
    1-=======-3
"""
l2 = cpw.line(50, 'mm')
DUT = concat_ports([l2, l2], port_order='second')
DUT.name = 'DUT'
DUT.nudge(1e-5)

## 构建测量系统* 级联左侧、DUT（待测设备）和右侧* 添加一些噪声

In [ ]:
"""
            Left        Meas         Right
             l1          l2           l1
    0----+-=======-2 0-=======-2 0-=======-+----2
         |                                 |
         = c1                              = c1
         |                                 |
        GND                               GND

             l1          l2           l1
    1----+-=======-3 1-=======-3 1-=======-+----3
         |                                 |
         = c1                              = c1
         |                                 |
        GND                               GND
"""
Meas = Left ** DUT ** Right
Meas.name = 'Meas'
Meas.add_noise_polar(1e-5, 2)

## 执行去嵌操作

In [ ]:
DUTd = Left.inv ** Meas ** Right.inv
DUTd.name = 'DUTd'

## 显示结果

In [ ]:

fig, axarr = plt.subplots(2,2, sharex=True, figsize=(10,6))

ax = axarr[0,0]
Meas.plot_s_db(m=0, n=0, ax=ax)
DUTd.plot_s_db(m=0, n=0, ax=ax)
DUT.plot_s_db(m=0, n=0, ax=ax, ls=':', color='0.0')
ax.set_title('Return Loss')
ax.legend(loc='lower center', ncol=3)
ax.grid(True)

ax = axarr[0,1]
Meas.plot_s_db(m=2, n=0, ax=ax)
DUTd.plot_s_db(m=2, n=0, ax=ax)
DUT.plot_s_db(m=2, n=0, ax=ax, ls=':', color='0.0')
ax.set_title('Insertion Loss')
ax.legend(loc='lower center', ncol=3)
ax.grid(True)

ax = axarr[1,0]
Meas.plot_s_db(m=1, n=0, ax=ax)
DUTd.plot_s_db(m=1, n=0, ax=ax)
DUT.plot_s_db(m=1, n=0, ax=ax, ls=':', color='0.0')
ax.set_title('Isolation')
ax.legend(loc='lower center', ncol=3)
ax.grid(True)

ax = axarr[1,1]
Meas.plot_s_deg(m=2, n=0, ax=ax)
DUTd.plot_s_deg(m=2, n=0, ax=ax, marker='o', markevery=25)
DUT.plot_s_deg(m=2, n=0, ax=ax, ls=':', color='0.0')
ax.set_title('Insertion Loss')
ax.legend(loc='lower center', ncol=3)
ax.grid(True)

fig.tight_layout()